# Creating an even centric knowledge graph

This notebook describes how an event-centric Knowledge Graph can be created by pushing capsules with event details to a GraphDB knowledge graph. Check out the README.md to get started. This notebooks assumed you installed the packages from the ```requirements.txt```in a separate conda environment and run this notebook in that environment.

## Dependencies imported

In [1]:
import json
import requests
from cltl.brain.long_term_memory import LongTermMemory
from cltl.commons.discrete import UtteranceType
from tqdm import tqdm
from random import getrandbits
from pathlib import Path
from dateutil import parser
from datetime import date, datetime
from tqdm import tqdm

## Setting up the Knowledge Graph end point

The README explain how to set up the Knowledge Graph in GraphDB. GraphDB needs to be running and a repository with the name ```event_sandbox``` need to be defined. In the notebook, we create a LongTermMemory instance, called ```brain``` that is used to push the conversational data and event triples to the repository. To create the instance, three parameters are required:

1. the address of the repository in GraphDB
2. the path to a folder (```graph```) to log the RDF triples that are pushed, which is created if not present
3. a boolean indicating if the repository needs to be emptied or not when calling

In [2]:
### Initialisation of the path for logging and of the GraDB repository for saving the data
# Create folders

graph_filepath = Path('./graph')
graph_filepath.mkdir(parents=True, exist_ok=True)

# Create brain connection
brain = LongTermMemory(address="http://localhost:7200/repositories/event_sandbox",  # Location to save accumulated graph
                       log_dir=graph_filepath,  # Location to save step-wise graphs
                       clear_all=True)  # To start from an empty brain

2026-06-03 08:50:47 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-06-03 08:50:47 -     INFO -                                    cltl.brain.LongTermMemory - Clearing brain
2026-06-03 08:50:47 -     INFO -                                    cltl.brain.LongTermMemory - Uploading ontology to brain
2026-06-03 08:50:48 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-06-03 08:50:48 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-06-03 08:50:48 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-06-03 08:50:48 -     INFO -                                   cltl.brain.TrustCalculator - Booted


When empty, the Knowledge Graph is first preloaded with a so-called T-box that is given in: [cltl-knowledgerepresentation](https://github.com/leolani/cltl-knowledgerepresentation/blob/main/src/cltl/brain/ontologies/integration.ttl)

This T-box combines various ontologies for repersenting the world, events and interactions, among others.

Given the T-box, interaction data can be pushed to the repository with details on the events that the interlocutors talk about.

## Populating the A-Box with interactions and event details

### Defining the context for the interaction

Every interaction consists of one or more interaction events that are part of a context. Before we can push interaction data to the Knowledge Graph, we first need to define an instance of a context of which the interaction is a subevent.

For this we use the ```capsule_context``` function which we give a dictionary (capsule) with the basic properties of the context and a unique identifier.

In [3]:
# Define contextual features
context_id = getrandbits(8)
place_id = getrandbits(8)
location = requests.get("https://ipinfo.io").json()
start_date = date(2021, 3, 12)

scenario_context = {"context_id": context_id,
                            "date": start_date,
                            "place": "Piek's office",
                            "place_id": place_id,
                            "country": location['country'],
                            "region": location['region'],
                            "city": location['city']}
brain.capsule_context(scenario_context)

2026-06-03 08:50:53 -     INFO -                                    cltl.brain.LongTermMemory - Context: context5


{'response': '204',
 'context': {'context_id': 5,
  'date': datetime.date(2021, 3, 12),
  'place': "Piek's office",
  'place_id': 103,
  'country': 'NL',
  'region': 'North Holland',
  'city': 'Amsterdam'},
 'rdf_log_path': PosixPath('graph/2026-06-03-08-50/brain_log_2026-06-03-08-50-53-792683')}

### Defining event-centric capsules for the turns in a conversation

Given the context, we can now create  a list of interaction events that we push to the repository to be added as subevents of the context. Each element in the list ```conversation_capsules``` that is shown below has a number of obligatory elements:

* [chat]: a unique identifier for the chat interaction
* [turn]: a unique identifier for the turn in the chat
* [author]: a triple dictionary that identifies the author
* [utterance]: the utterance itself
* [utterance_type]: the kind of utterance
* [position]: the offset position that marks the expression(s) in the utterance that mention the information 
* [event_details]: the information as a list of properties of events, where the subject identifies the event
* [perspective]: the perspective of the author (the source) on the event details as a claim
* [timestamp]: when the utterance took place
* [context_id]: the context identifier that was pushed previously.

In [13]:
conversation_capsules = [{"chat": 12,
 "turn": 1,
 "author": {"label": "piek", "type": ["person"], 'uri': "http://cltl.nl/leolani/friends/piek-1"},
 "utterance": "John drinks beer",
 "utterance_type": UtteranceType.STATEMENT,
 "position": "0-15",
 "event_details": [
     {
     "subject": {"label": "drink", "type": ["event"],
                 "uri": "http://cltl.nl/leolani/world/drink-2"
                 },
     "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
     "object": {"label": "john", "type": ["person"],
                "uri": "http://cltl.nl/leolani/world/john-1"
                }

     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
         "object": {"label": "beer", "type": ["drink"],
                    "uri": "http://cltl.nl/leolani/world/beer"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasPlace", "uri": "sem:hasPlace"},
         "object": {"label": "pub", "type": ["place"],
                    "uri": "http://cltl.nl/leolani/world/pub"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasTime", "uri": "sem:hasTime"},
         "object": {"label": "8-3-2026", "type": ["date"],
                    "uri": ""}
     }
 ],
 "perspective": {
     "certainty": 1,
     "polarity": 1,
     "sentiment": 1
 },
 "timestamp": datetime.combine(start_date, datetime.now().time()),
 "context_id": context_id
 },
{"chat": 12,
 "turn": 2,
 "author": {"label": "luis", "type": ["person"], 'uri': "http://cltl.nl/leolani/friends/luis-1"},
 "utterance": "John does not drink beer",
 "utterance_type": UtteranceType.STATEMENT,
 "position": "0-15",
 "event_details": [
     {
     "subject": {"label": "drink", "type": ["event"],
                 "uri": "http://cltl.nl/leolani/world/drink-2"
                 },
     "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
     "object": {"label": "john", "type": ["person"],
                "uri": "http://cltl.nl/leolani/world/john-1"
                }

     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasActor", "uri": "sem:hasActor"},
         "object": {"label": "beer", "type": ["drink"],
                    "uri": "http://cltl.nl/leolani/world/beer"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasPlace", "uri": "sem:hasPlace"},
         "object": {"label": "pub", "type": ["place"],
                    "uri": "http://cltl.nl/leolani/world/pub"}
     },
     {
         "subject": {"label": "drink", "type": ["event"],
                     "uri": "http://cltl.nl/leolani/world/drink-2"},
         "predicate": {"label": "hasTime", "uri": "sem:hasTime"},
         "object": {"label": "8-3-2026", "type": ["date"],
                    "uri": ""}
     }
 ],
 "perspective": {
     "certainty": 1,
     "polarity": -1,
     "sentiment": -1
 },
 "timestamp": datetime.combine(start_date, datetime.now().time()),
 "context_id": context_id
 }]

The event details create an event-centric Knowledge graph. This means that the subject defines an instance of an event through a unique URI, while the predicate and object express values for properties of the event. We follow here the Simple Event Ontology (SEM, van Hage et al 2011) that is also part of the T-box. Typically, the predicates are semantic roles that define the event. In SEM, these are hasActor, hasPlace and hasTime but also more specific semantic roles can be defined. It is advised to define these specific roles as subclasses of the SEM predicates.

### Pushing the turn capsules to the Graph

The above conversational data is pushed to the repository. It will add the interaction events as such, i.e. chats, turns, but also the events reported. All as part of the context.

In [9]:
for capsule in conversation_capsules:
    brain.capsule_event(capsule, reason_types=True, return_thoughts=False, create_label=True)

2026-06-03 08:57:29 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_agent_Jan [activity or exercise_->_agent])
2026-06-03 08:57:29 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_location_Dutch [activity or exercise_->_place])
2026-06-03 08:57:29 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_time_lately [activity or exercise_->_time])
2026-06-03 08:57:29 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_agent_Jan [activity or exercise_->_agent])
2026-06-03 08:57:29 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_time_lately [activity or exercise_->_time])
2026-06-03 08:57:30 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_agent_Jan [activity or exerc

You can now inspect the event_sandbox in GraphDB to find the event data, which should look as follows.

In [16]:
from IPython.display import Image
Image(url="ekg-notebook.png", width=800, height=600)

## Multiple conversations and events in time

In the next part, we are pushing the triples for two conversations to the KG. We assume that there are two moments in time that an agent has a conversation with a person ```Jan```:

* chat:0 at 2013-04-13
* chat:1 at 2013-05-11

We will therefore create two separate contexts for each date as well and push these to the KG to save the conversations as subevents. For each turn in each chat, we therefore specify the corresponding context identifier.

Before, we start, we will reinitialise the ```event_sandbox``` and remove the previous data.

In [19]:
# Create brain connection
brain = LongTermMemory(address="http://localhost:7200/repositories/event_sandbox",  # Location to save accumulated graph
                       log_dir=graph_filepath,  # Location to save step-wise graphs
                       clear_all=True)  # To start from an empty brain

2026-06-03 10:35:24 -     INFO -                                    cltl.brain.LongTermMemory - Booted
2026-06-03 10:35:24 -     INFO -                                    cltl.brain.LongTermMemory - Clearing brain
2026-06-03 10:35:25 -     INFO -                                    cltl.brain.LongTermMemory - Uploading ontology to brain
2026-06-03 10:35:25 -     INFO -                                  cltl.brain.ThoughtGenerator - Booted
2026-06-03 10:35:25 -     INFO -                                  cltl.brain.LocationReasoner - Booted
2026-06-03 10:35:25 -     INFO -                                      cltl.brain.TypeReasoner - Booted
2026-06-03 10:35:25 -     INFO -                                   cltl.brain.TrustCalculator - Booted


In [20]:
# Define contextual features
context0_id = getrandbits(8)
place_id = getrandbits(8)
location = requests.get("https://ipinfo.io").json()
start_date = date(2013, 4, 13)

scenario_context = {"context_id": context0_id,
                            "date": start_date,
                            "place": "Piek's office",
                            "place_id": place_id,
                            "country": location['country'],
                            "region": location['region'],
                            "city": location['city']}
brain.capsule_context(scenario_context)

2026-06-03 10:35:28 -     INFO -                                    cltl.brain.LongTermMemory - Context: context64


{'response': '204',
 'context': {'context_id': 64,
  'date': datetime.date(2013, 4, 13),
  'place': "Piek's office",
  'place_id': 203,
  'country': 'NL',
  'region': 'North Holland',
  'city': 'Amsterdam'},
 'rdf_log_path': PosixPath('graph/2026-06-03-10-35/brain_log_2026-06-03-10-35-28-844384')}

The next capsules represent the turns of the first conversation. In this example, we added richer specifications for the perspective and the event details have more specific semantic roles as well type information for each activity.

In [21]:
#### Conversation 0 capsules added to context0_id

conversation0_capsules = [    
    {
        "chat": 0,
        "turn": 1,
        "author": {
            "label": "agent",
            "type": [
                "agent"
            ],
            "uri": "http://cltl.nl/leolani/friends/agent"
        },
        "utterance": "Hey Jan, I see that you're a fan of cycling. Has the Dutch weather been favorable for it lately?",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-96",
        "perspective": {
            "certainty": "UNDERSPECIFIED",
            "polarity": "POSITIVE",
            "sentiment": "POSITIVE",
            "emotion": [
                "CURIOSITY"
            ],
            "level": "SOMEWHAT"
        },
        "timestamp": "2013-04-13 21:44:12.163845",
        "context_id": context0_id,
        "event_details": [
            {
                "subject": {
                    "label": "cycling",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/cycling0.6598833213731631"
                },
                "predicate": {
                    "label": "agent",
                    "uri": "http://cltl.nl/leolani/n2mu/agent"
                },
                "object": {
                    "label": "Jan",
                    "type": [
                        "agent"
                    ],
                    "uri": ""
                }
            },
            {
                "subject": {
                    "label": "cycling",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/cycling0.6598833213731631"
                },
                "predicate": {
                    "label": "location",
                    "uri": "http://cltl.nl/leolani/n2mu/location"
                },
                "object": {
                    "label": "Dutch",
                    "type": [
                        "place"
                    ],
                    "uri": ""
                }
            },
            {
                "subject": {
                    "label": "cycling",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/cycling0.6598833213731631"
                },
                "predicate": {
                    "label": "time",
                    "uri": "http://cltl.nl/leolani/n2mu/time"
                },
                "object": {
                    "label": "lately",
                    "type": [
                        "time"
                    ],
                    "uri": ""
                }
            }
        ]
    },
    {
        "chat": 0,
        "turn": 2,
        "author": {
            "label": "Jan",
            "type": [
                "agent"
            ],
            "uri": "http://cltl.nl/leolani/friends/Jan"
        },
        "utterance": "It's been quite unpredictable recently, so I haven't been out as often as I'd like. How does this affect my diabetes management?",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-128",
        "perspective": {
            "certainty": "UNDERSPECIFIED",
            "polarity": "POSITIVE",
            "sentiment": "NEGATIVE",
            "emotion": [
                "CURIOSITY"
            ],
            "level": "SOMEWHAT"
        },
        "timestamp": "2013-04-13 21:44:12.427143",
        "context_id": context0_id,
        "event_details": [
            {
                "subject": {
                    "label": "cycling",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/cycling0.6598833213731631"
                },
                "predicate": {
                    "label": "agent",
                    "uri": "http://cltl.nl/leolani/n2mu/agent"
                },
                "object": {
                    "label": "Jan",
                    "type": [
                        "agent"
                    ],
                    "uri": ""
                }
            },
            {
                "subject": {
                    "label": "cycling",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/cycling0.6598833213731631"
                },
                "predicate": {
                    "label": "time",
                    "uri": "http://cltl.nl/leolani/n2mu/time"
                },
                "object": {
                    "label": "lately",
                    "type": [
                        "time"
                    ],
                    "uri": ""
                }
            }
        ],
    },
    {
        "chat": 0,
        "turn": 3,
        "author": {
            "label": "agent",
            "type": [
                "agent"
            ],
            "uri": "http://cltl.nl/leolani/friends/agent"
        },
        "utterance": "Physical activity, like cycling, is crucial in managing your diabetes. It helps your body use insulin better and can lower your blood sugar levels. If weather conditions are interfering with your routine, perhaps you could look for indoor exercises to stay active.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-264",
        "perspective": {
            "certainty": "UNDERSPECIFIED",
            "polarity": "POSITIVE",
            "sentiment": "POSITIVE",
            "emotion": [
                "UNDERSPECIFIED"
            ],
            "level": "ALMOST_FULLY"
        },
        "timestamp": "2013-04-13 21:44:12.709687",
        "context_id": context0_id,
        "event_details": [
            {
                "subject": {
                    "label": "cycling",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/cycling0.6598833213731631"
                },
                "predicate": {
                    "label": "agent",
                    "uri": "http://cltl.nl/leolani/n2mu/agent"
                },
                "object": {
                    "label": "Jan",
                    "type": [
                        "agent"
                    ],
                    "uri": ""
                }
            }
        ]
    }
]

We now add the capsules for each turn to the context0.

In [22]:
for capsule in conversation0_capsules:
    brain.capsule_event(capsule, reason_types=True, return_thoughts=False, create_label=True)

2026-06-03 10:35:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_agent_Jan [activity or exercise_->_agent])
2026-06-03 10:35:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_location_Dutch [activity or exercise_->_place])
2026-06-03 10:35:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_time_lately [activity or exercise_->_time])
2026-06-03 10:35:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_agent_Jan [activity or exercise_->_agent])
2026-06-03 10:35:48 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_time_lately [activity or exercise_->_time])
2026-06-03 10:35:49 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: cycling_agent_Jan [activity or exerc

In [11]:
from IPython.display import Image
Image(url="ekg-notebook2.png", width=800, height=600)

Next, we define the second context for the conversation that takes place on 11-05-2023. We also give it a different location. Note that we do not reinitialse the LongTermMemory so that the new context and the conversation will be added to the previous data.

In [23]:
# Define contextual features
context1_id = getrandbits(8)
place_id = getrandbits(8)
location = requests.get("https://ipinfo.io").json()
start_date = date(2013, 5, 11)

scenario_context = {"context_id": context1_id,
                            "date": start_date,
                            "place": "VU Aula",
                            "place_id": place_id,
                            "country": location['country'],
                            "region": location['region'],
                            "city": location['city']}
brain.capsule_context(scenario_context)

2026-06-03 10:36:52 -     INFO -                                    cltl.brain.LongTermMemory - Context: context184


{'response': '204',
 'context': {'context_id': 184,
  'date': datetime.date(2013, 5, 11),
  'place': 'VU Aula',
  'place_id': 103,
  'country': 'NL',
  'region': 'North Holland',
  'city': 'Amsterdam'},
 'rdf_log_path': PosixPath('graph/2026-06-03-10-35/brain_log_2026-06-03-10-36-52-305567')}

In [28]:
conversation1_capsules = [
 
    {
        "chat": 1,
        "turn": 7,
        "author": {
            "label": "agent",
            "type": [
                "agent"
            ],
            "uri": "http://cltl.nl/leolani/friends/agent"
        },
        "utterance": "Yes, absolutely! Regular walking is an efficient and effective way to improve insulin sensitivity and manage your diabetes. Just make sure to adapt the intensity and duration to your abilities.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-193",
        "perspective": {
            "certainty": "PROBABLE",
            "polarity": "POSITIVE",
            "sentiment": "POSITIVE",
            "emotion": [
                "APPROVAL"
            ],
            "level": "SOMEWHAT"
        },
        "timestamp": "2013-05-11 17:24:19.8644",
        "context_id": context1_id,
        "event_details": [
            {
                "subject": {
                    "label": "regular walking",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/regular_walking0.27740476088782606"
                },
                "predicate": {
                    "label": "agent",
                    "uri": "http://cltl.nl/leolani/n2mu/agent"
                },
                "object": {
                    "label": "Jan",
                    "type": [
                        "agent"
                    ],
                    "uri": ""
                }
            },
            {
                "subject": {
                    "label": "regular walking",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/regular_walking0.27740476088782606"
                },
                "predicate": {
                    "label": "manner",
                    "uri": "http://cltl.nl/leolani/n2mu/manner"
                },
                "object": {
                    "label": "adapt intensity and duration",
                    "type": [
                        "property"
                    ],
                    "uri": ""
                }
            },
            {
                "subject": {
                    "label": "regular walking",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/regular_walking0.27740476088782606"
                },
                "predicate": {
                    "label": "time",
                    "uri": "http://cltl.nl/leolani/n2mu/time"
                },
                "object": {
                    "label": "daily",
                    "type": [
                        "time"
                    ],
                    "uri": ""
                }
            }
        ]
    },
    {
        "chat": 1,
        "turn": 8,
        "author": {
            "label": "Jan",
            "type": [
                "agent"
            ],
            "uri": "http://cltl.nl/leolani/friends/Jan"
        },
        "utterance": "I got it. Thanks for your advice. I will try to keep active, rain or shine.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-75",
        "perspective": {
            "certainty": "UNDERSPECIFIED",
            "polarity": "POSITIVE",
            "sentiment": "POSITIVE",
            "emotion": [
                "GRATITUDE"
            ],
            "level": "SOMEWHAT"
        },
        "timestamp": "2013-05-11 17:44:14.480872",
        "context_id": context1_id,
        "event_details": [
            {
                "subject": {
                    "label": "walking",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/walking0.2987234562034081"
                },
                "predicate": {
                    "label": "agent",
                    "uri": "http://cltl.nl/leolani/n2mu/agent"
                },
                "object": {
                    "label": "Jan",
                    "type": [
                        "agent"
                    ],
                    "uri": ""
                }
            },
            {
                "subject": {
                    "label": "walking",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/walking0.2987234562034081"
                },
                "predicate": {
                    "label": "time",
                    "uri": "http://cltl.nl/leolani/n2mu/time"
                },
                "object": {
                    "label": "daily",
                    "type": [
                        "time"
                    ],
                    "uri": ""
                }
            }
        ]
    },
    {
        "chat": 1,
        "turn": 9,
        "author": {
            "label": "agent",
            "type": [
                "agent"
            ],
            "uri": "http://cltl.nl/leolani/friends/agent"
        },
        "utterance": "You're welcome, Jan! Remember, consistency is key in managing diabetes. Feel free to reach out if you need any more advice.",
        "utterance_type": UtteranceType.STATEMENT,
        "position": "0-123",
        "perspective": {
            "certainty": "UNDERSPECIFIED",
            "polarity": "POSITIVE",
            "sentiment": "POSITIVE",
            "emotion": [
                "GRATITUDE"
            ],
            "level": "SOMEWHAT"
        },
        "timestamp": "2013-05-11 17:44:15.380442",
        "context_id": context1_id,
        "event_details": [
            {
                "subject": {
                    "label": "daily walks",
                    "type": [
                        "activity",
                        "exercise"
                    ],
                    "uri": "http://cltl.nl/leolani/n2mu/daily_walks0.5581357206184595"
                },
                "predicate": {
                    "label": "agent",
                    "uri": "http://cltl.nl/leolani/n2mu/agent"
                },
                "object": {
                    "label": "Jan",
                    "type": [
                        "agent"
                    ],
                    "uri": ""
                }
            }
        ]
    }
]

In [29]:
for capsule in conversation1_capsules:
    brain.capsule_event(capsule, reason_types=True, return_thoughts=False, create_label=True)

2026-06-03 10:38:03 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: regular walking_agent_Jan [activity or exercise_->_agent])
2026-06-03 10:38:03 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: regular walking_manner_adapt intensity and duration [activity or exercise_->_property])
2026-06-03 10:38:03 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: regular walking_time_daily [activity or exercise_->_time])
2026-06-03 10:38:04 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: walking_agent_Jan [activity or exercise_->_agent])
2026-06-03 10:38:04 -     INFO -                                    cltl.brain.LongTermMemory - Triple in statement: walking_time_daily [activity or exercise_->_time])
2026-06-03 10:38:04 -     INFO -                                    cltl.brain.LongTermMemory - Triple in s

In the KG, we can visualise the triples involving Jan and see now several events and the utterances of two chats as shown in the next image.

In [30]:
from IPython.display import Image
Image(url="ekg-notebook3.png", width=800, height=600)

## SPARQL to extract timeline data

The next SPARQL query shows how activities, represented by the text expressions that mentions it, can be extracted from the KG in which ```Jan``` is denoted. The time of the activity is here based on the time of the utterance. The query also extracts the source of the utterance and the perspective of the source on the activity.

```
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sem: <http://semanticweb.cs.vu.nl/2009/11/sem/>
PREFIX n2mu: <http://cltl.nl/leolani/n2mu/>
PREFIX gaf: <http://groundedannotationframework.org/gaf#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX grasp: <http://groundedannotationframework.org/grasp#>
PREFIX graspl: <http://groundedannotationframework.org/grasp/level#>
PREFIX leolaniWorld: <http://cltl.nl/leolani/world/>
select ?time ?activity ?source ?perspective where {
    ?a a n2mu:activity .
    ?a rdfs:label ?activity .
    ?a gaf:denotedIn ?u.
    leolaniWorld:Jan gaf:denotedIn ?u.
    ?u grasp:hasAttribution ?o .
    ?u sem:hasBeginTimeStamp ?time .
    ?u grasp:wasAttributedTo ?source .
	?o rdf:value ?perspective
} 
```

The results for querying the above KG are saved in the query-result.csv. ```The plot_activity_timeline.py``` reads the results and generates a timeline plot.

In [3]:
from IPython.display import Image
Image(url="activity_timeline.png", width=800, height=600)